# 🧠 Task-4: Binary Classification System
### AI/ML Internship Project

**Dataset:** Breast Cancer Dataset (scikit-learn built-in)  
**Problem:** Classify tumors as Malignant (1) or Benign (0)  
**Goal:** Build, evaluate, and compare classification models properly

---
## 📋 What We Will Cover
1. Load & Explore the Dataset
2. Preprocessing & Train-Test Split
3. Train Multiple Classification Models
4. Evaluate Using Proper Metrics (not just accuracy)
5. Handle Class Imbalance
6. Compare Models & Select Best One
7. Final Summary & Justification

---
## Step 0: Install / Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset
from sklearn.datasets import load_breast_cancer

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)

# Class imbalance handling
from sklearn.utils.class_weight import compute_class_weight
# pip install imbalanced-learn  <-- run this once in terminal if needed
from imblearn.over_sampling import SMOTE

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ All libraries imported successfully!')

---
## Step 1: Load & Explore the Dataset

In [ ]:
# Load the Breast Cancer dataset
cancer = load_breast_cancer()

# Create a DataFrame for easy exploration
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target  # 0 = Malignant, 1 = Benign

print('📊 Dataset Shape:', df.shape)
print('\n🎯 Target Classes:', cancer.target_names)  # ['malignant', 'benign']
print('\n📝 First 5 Rows:')
df.head()

In [ ]:
# Basic info about the dataset
print('=== Dataset Info ===')
print(f'Total Samples   : {df.shape[0]}')
print(f'Total Features  : {df.shape[1] - 1}')
print(f'Missing Values  : {df.isnull().sum().sum()}')
print()

# Class distribution
print('=== Class Distribution ===')
class_counts = df['target'].value_counts()
print(f'Benign    (1): {class_counts[1]} samples ({class_counts[1]/len(df)*100:.1f}%)')
print(f'Malignant (0): {class_counts[0]} samples ({class_counts[0]/len(df)*100:.1f}%)')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(['Malignant (0)', 'Benign (1)'], class_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Malignant (0)', 'Benign (1)'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0.05, 0.05), shadow=True)
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('📊 Dataset Class Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Note: Dataset is slightly imbalanced (63% Benign vs 37% Malignant)')

In [ ]:
# Statistical summary
print('=== Statistical Summary (first 5 features) ===')
df.iloc[:, :5].describe().round(2)

In [ ]:
# Correlation heatmap of top features
top_features = df.columns[:10].tolist() + ['target']
plt.figure(figsize=(12, 8))
corr = df[top_features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5)
plt.title('🔥 Feature Correlation Heatmap (Top 10 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 2: Preprocessing & Train-Test Split

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-Test Split (80-20), stratified to maintain class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('=== Data Split ===')
print(f'Training Set  : {X_train.shape[0]} samples')
print(f'Testing Set   : {X_test.shape[0]} samples')
print()
print('=== Class Distribution After Split ===')
print(f'Train - Benign: {sum(y_train==1)}, Malignant: {sum(y_train==0)}')
print(f'Test  - Benign: {sum(y_test==1)}, Malignant: {sum(y_test==0)}')

In [ ]:
# Feature Scaling (important for distance-based models like KNN, SVM, Logistic Regression)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # Fit on TRAIN only, then transform
X_test_scaled  = scaler.transform(X_test)         # Only transform test (NO fitting!)

print('✅ Feature scaling done!')
print(f'Mean of scaled train data (should be ~0): {X_train_scaled.mean():.4f}')
print(f'Std  of scaled train data (should be ~1): {X_train_scaled.std():.4f}')

---
## Step 3: Train Multiple Classification Models

In [ ]:
# Define all models to compare
models = {
    'Logistic Regression'     : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'           : DecisionTreeClassifier(random_state=42),
    'Random Forest'           : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'       : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM'                     : SVC(probability=True, random_state=42),
    'K-Nearest Neighbors'     : KNeighborsClassifier(n_neighbors=5)
}

print(f'✅ {len(models)} models defined and ready to train!')

In [ ]:
# Train all models and collect predictions
results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]  # Probability for positive class
    
    # Calculate metrics
    results[name] = {
        'model'    : model,
        'y_pred'   : y_pred,
        'y_prob'   : y_prob,
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'f1'       : f1_score(y_test, y_pred),
        'roc_auc'  : roc_auc_score(y_test, y_prob)
    }
    print(f'✅ {name} trained!')

print('\n🎉 All models trained!')

---
## Step 4: Evaluate Using Proper Metrics

### Why accuracy alone is NOT enough?
- If 90% of patients are Benign, a model that always predicts Benign gets **90% accuracy** but is **useless**
- In medical diagnosis: **Missing a Malignant tumor is FAR worse** than a false alarm
- We need **Recall, Precision, F1-Score, and ROC-AUC**

| Metric | What it measures |
|--------|------------------|
| **Accuracy** | % of correct predictions overall |
| **Precision** | Of all predicted positive, how many are actually positive? |
| **Recall** | Of all actual positive, how many did we catch? (very important in medical) |
| **F1-Score** | Harmonic mean of Precision & Recall |
| **ROC-AUC** | Overall ability to distinguish classes (1.0 = perfect) |

In [ ]:
# Build comparison table
metrics_df = pd.DataFrame({
    name: {
        'Accuracy' : round(r['accuracy'], 4),
        'Precision': round(r['precision'], 4),
        'Recall'   : round(r['recall'], 4),
        'F1-Score' : round(r['f1'], 4),
        'ROC-AUC'  : round(r['roc_auc'], 4)
    }
    for name, r in results.items()
}).T

# Sort by F1-Score
metrics_df = metrics_df.sort_values('F1-Score', ascending=False)

print('=== 📊 Model Comparison Table ===')
print('(Sorted by F1-Score)')
metrics_df

In [ ]:
# Visualize all metrics for all models
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_df.index))
width = 0.15

fig, ax = plt.subplots(figsize=(16, 7))
colors_list = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors_list)):
    offset = (i - 2) * width
    bars = ax.bar(x + offset, metrics_df[metric], width, label=metric,
                  color=color, alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('📊 Model Comparison: All Evaluation Metrics', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df.index, rotation=15, ha='right', fontsize=10)
ax.set_ylim(0.8, 1.02)
ax.legend(loc='lower right', fontsize=10)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, r) in enumerate(results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Malignant', 'Benign'])
    disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')
    axes[idx].set_title(f'{name}\nF1={r["f1"]:.3f} | AUC={r["roc_auc"]:.3f}',
                        fontsize=11, fontweight='bold')

plt.suptitle('🔍 Confusion Matrices — All Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("""
📖 Reading the Confusion Matrix:
  Top-Left     (TN): Correctly predicted Malignant
  Top-Right    (FP): Predicted Benign but actually Malignant ← DANGEROUS in medical!
  Bottom-Left  (FN): Predicted Malignant but actually Benign
  Bottom-Right (TP): Correctly predicted Benign
""")

In [ ]:
# ROC Curve for all models
plt.figure(figsize=(10, 8))

colors_roc = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{name} (AUC = {r["roc_auc"]:.3f})')

# Random classifier baseline
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.500)')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('📈 ROC Curves — All Models', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('💡 Higher AUC = Better model. Curve closer to top-left corner = better.')

In [ ]:
# Detailed Classification Report for the best model (by F1)
best_model_name = metrics_df.index[0]
best_result = results[best_model_name]

print(f'🏆 Best Model: {best_model_name}')
print('='*50)
print(classification_report(y_test, best_result['y_pred'],
                             target_names=['Malignant', 'Benign']))

---
## Step 5: Handle Class Imbalance

### What is Class Imbalance?
- Our dataset has ~63% Benign vs ~37% Malignant — **moderately imbalanced**
- In real scenarios (fraud, churn), imbalance can be 99% vs 1%
- Techniques:
  1. **Class Weights** — Penalize wrong predictions on minority class more
  2. **SMOTE** — Synthetic Minority Over-sampling Technique (creates synthetic samples)
  3. **Undersampling** — Reduce majority class (not used here, we lose data)

In [ ]:
# ── Method 1: Class Weights ──
# Logistic Regression with class_weight='balanced'

lr_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_balanced.fit(X_train_scaled, y_train)
y_pred_balanced = lr_balanced.predict(X_test_scaled)
y_prob_balanced = lr_balanced.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression: Default vs Class-Weighted ===')
print(f"{'Metric':<12} {'Default':>10} {'Balanced':>10}")
print('-'*35)
lr_default = results['Logistic Regression']
for metric, func in [('Accuracy', accuracy_score), ('Recall', lambda y,p: recall_score(y,p)),
                      ('Precision', lambda y,p: precision_score(y,p)), ('F1', lambda y,p: f1_score(y,p))]:
    d = func(y_test, lr_default['y_pred'])
    b = func(y_test, y_pred_balanced)
    print(f"{metric:<12} {d:>10.4f} {b:>10.4f}")

print('\n💡 With class_weight="balanced", model is forced to focus more on minority class')

In [ ]:
# ── Method 2: SMOTE ──
# Create synthetic samples for the minority class

print('Before SMOTE:')
print(f'  Malignant (0): {sum(y_train==0)}')
print(f'  Benign    (1): {sum(y_train==1)}')

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print('\nAfter SMOTE:')
print(f'  Malignant (0): {sum(y_train_smote==0)}')
print(f'  Benign    (1): {sum(y_train_smote==1)}')

# Train Random Forest on SMOTE data
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = rf_smote.predict(X_test_scaled)

print(f'\n=== Random Forest: Default vs SMOTE ===')
print(f"{'Metric':<12} {'Default':>10} {'SMOTE':>10}")
print('-'*35)
rf_default = results['Random Forest']
for metric, func in [('Accuracy', accuracy_score),
                      ('Recall', lambda y,p: recall_score(y,p)),
                      ('Precision', lambda y,p: precision_score(y,p)),
                      ('F1', lambda y,p: f1_score(y,p))]:
    d = func(y_test, rf_default['y_pred'])
    s = func(y_test, y_pred_smote)
    print(f"{metric:<12} {d:>10.4f} {s:>10.4f}")

In [ ]:
# Visualize SMOTE effect
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before SMOTE
before_counts = [sum(y_train==0), sum(y_train==1)]
axes[0].bar(['Malignant (0)', 'Benign (1)'], before_counts,
            color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Before SMOTE', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Samples')
for i, v in enumerate(before_counts):
    axes[0].text(i, v+2, str(v), ha='center', fontsize=12, fontweight='bold')

# After SMOTE
after_counts = [sum(y_train_smote==0), sum(y_train_smote==1)]
axes[1].bar(['Malignant (0)', 'Benign (1)'], after_counts,
            color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1].set_title('After SMOTE', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Samples')
for i, v in enumerate(after_counts):
    axes[1].text(i, v+2, str(v), ha='center', fontsize=12, fontweight='bold')

plt.suptitle('⚖️ Class Balance: Before vs After SMOTE', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6: Cross-Validation for Reliable Evaluation

Cross-validation gives a more reliable performance estimate than a single train-test split.

In [ ]:
# Stratified K-Fold Cross-Validation (5-fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

for name, model in models.items():
    cv_f1  = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='f1')
    cv_auc = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='roc_auc')
    cv_results[name] = {
        'CV F1 Mean' : cv_f1.mean(),
        'CV F1 Std'  : cv_f1.std(),
        'CV AUC Mean': cv_auc.mean(),
        'CV AUC Std' : cv_auc.std()
    }

cv_df = pd.DataFrame(cv_results).T.round(4)
cv_df = cv_df.sort_values('CV F1 Mean', ascending=False)
print('=== 5-Fold Cross-Validation Results ===')
cv_df

In [ ]:
# Visualize cross-validation results with error bars
fig, ax = plt.subplots(figsize=(12, 6))

models_list = cv_df.index.tolist()
means = cv_df['CV F1 Mean'].values
stds  = cv_df['CV F1 Std'].values

bars = ax.barh(models_list, means, xerr=stds, color='#3498db',
               alpha=0.8, edgecolor='black', capsize=5, height=0.5)

for bar, mean, std in zip(bars, means, stds):
    ax.text(mean + std + 0.002, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f} ± {std:.3f}', va='center', fontsize=10)

ax.set_xlabel('F1 Score (mean ± std)', fontsize=12)
ax.set_title('📊 Cross-Validation F1 Scores (5-Fold)', fontsize=14, fontweight='bold')
ax.set_xlim(0.8, 1.08)
ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('💡 Lower std = more stable/reliable model')

---
## Step 7: Feature Importance (for the Best Model)

In [ ]:
# Feature Importance from Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=cancer.feature_names
).sort_values(ascending=False)

# Plot top 15 features
plt.figure(figsize=(12, 7))
top15 = feature_importance.head(15)
colors_feat = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top15)))

bars = plt.barh(range(len(top15)), top15.values[::-1],
                color=colors_feat, edgecolor='black', alpha=0.85)
plt.yticks(range(len(top15)), top15.index[::-1], fontsize=10)
plt.xlabel('Feature Importance Score', fontsize=12)
plt.title('🌲 Random Forest — Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

for i, (bar, val) in enumerate(zip(bars, top15.values[::-1])):
    plt.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## Step 8: Final Summary & Model Justification

In [ ]:
# Final Summary Table combining test metrics + CV results
summary = metrics_df.copy()
summary['CV F1 Mean'] = cv_df['CV F1 Mean']
summary['CV F1 Std']  = cv_df['CV F1 Std']
summary = summary.sort_values('F1-Score', ascending=False)

print('='*75)
print('                   🏆 FINAL MODEL COMPARISON SUMMARY')
print('='*75)
print(summary.to_string())
print('='*75)

winner = summary.index[0]
print(f'\n🥇 Best Model: {winner}')
print(f'   Test F1-Score  : {summary.loc[winner, "F1-Score"]:.4f}')
print(f'   Test ROC-AUC   : {summary.loc[winner, "ROC-AUC"]:.4f}')
print(f'   Test Recall    : {summary.loc[winner, "Recall"]:.4f}')
print(f'   CV F1 Mean±Std : {summary.loc[winner, "CV F1 Mean"]:.4f} ± {summary.loc[winner, "CV F1 Std"]:.4f}')

In [ ]:
# Final visualization — radar chart style comparison
fig, ax = plt.subplots(figsize=(10, 7))

# Heatmap of all metrics
heatmap_data = summary[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='YlOrRd',
            vmin=0.8, vmax=1.0, linewidths=0.5, ax=ax, annot_kws={'size': 11})
ax.set_title('🎯 Model Performance Heatmap (All Metrics)', fontsize=14, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║           📋 TASK-4 LEARNING OUTCOMES — SUMMARY                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  ✅ 1. Classification Problem                                        ║
║        Breast cancer = binary classification (Malignant vs Benign)  ║
║                                                                      ║
║  ✅ 2. Multiple Models Trained                                       ║
║        Logistic Regression, Decision Tree, Random Forest,           ║
║        Gradient Boosting, SVM, KNN                                  ║
║                                                                      ║
║  ✅ 3. Evaluation Beyond Accuracy                                    ║
║        Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix       ║
║                                                                      ║
║  ✅ 4. Class Imbalance Handled                                       ║
║        class_weight='balanced' and SMOTE applied                    ║
║                                                                      ║
║  ✅ 5. Scientific Justification                                      ║
║        5-Fold Cross-Validation used for reliable evaluation         ║
║        Feature importance analyzed                                   ║
║        Best model selected based on F1 + ROC-AUC + CV stability     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")